# Milestone 4: Virtual Screening Pipeline
Batch-score a ZINC compound library through the multi-target toxicity model.
Ranks candidates by composite risk score and filters by drug-likeness.

In [ ]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
warnings.filterwarnings('ignore')
print("Imports OK")

In [ ]:
BASE_MODEL = 'DeepChem/ChemBERTa-77M-MTR'
checkpoints = sorted(glob.glob('chemberta-tox21-multitarget-*'))
MODEL_DIR = checkpoints[-1]
print(f"Checkpoint: {MODEL_DIR}")

NUM_TARGETS = 12
TARGET_NAMES = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
]
THRESHOLDS = {
    'NR-AR': 0.95, 'NR-AR-LBD': 0.95, 'NR-AhR': 0.75,
    'NR-Aromatase': 0.85, 'NR-ER': 0.70, 'NR-ER-LBD': 0.85,
    'NR-PPAR-gamma': 0.85, 'SR-ARE': 0.65, 'SR-ATAD5': 0.80,
    'SR-HSE': 0.85, 'SR-MMP': 0.85, 'SR-p53': 0.85,
}
SEVERITY_WEIGHTS = {
    'NR-AR': 1.0, 'NR-AR-LBD': 1.0, 'NR-AhR': 1.5,
    'NR-Aromatase': 1.0, 'NR-ER': 1.0, 'NR-ER-LBD': 1.0,
    'NR-PPAR-gamma': 1.0, 'SR-ARE': 1.0, 'SR-ATAD5': 1.0,
    'SR-HSE': 1.0, 'SR-MMP': 1.5, 'SR-p53': 1.5,
}
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=NUM_TARGETS,
    ignore_mismatched_sizes=True, attn_implementation='eager',
)
model = PeftModel.from_pretrained(base, MODEL_DIR).merge_and_unload()
model.eval().to(DEVICE)
print("Model ready.")

In [ ]:
ZINC_URL = (
    "https://raw.githubusercontent.com/aspuru-guzik-group/"
    "chemical_vae/master/models/zinc_properties/"
    "250k_rndm_zinc_drugs_clean_3.csv"
)
CACHE = "data/zinc_subset.csv"
N_SAMPLE = 5000

if os.path.exists(CACHE):
    df_raw = pd.read_csv(CACHE)
    print(f"Loaded {len(df_raw)} compounds from cache.")
else:
    os.makedirs("data", exist_ok=True)
    print("Downloading ZINC-250k...")
    df_full = pd.read_csv(ZINC_URL)
    df_raw = df_full.sample(N_SAMPLE, random_state=42).reset_index(drop=True)
    df_raw.to_csv(CACHE, index=False)
    print(f"Downloaded and cached {len(df_raw)} compounds.")

print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# === ASSERTION CELL ===
mols = [Chem.MolFromSmiles(s) for s in df_raw['smiles']]
valid_mask = [m is not None for m in mols]
valid_smiles = df_raw['smiles'][valid_mask].tolist()
valid_mols   = [m for m in mols if m is not None]

assert len(valid_smiles) >= 4500, f"Expected ≥4500 valid SMILES, got {len(valid_smiles)}"
print(f"✓ {len(valid_smiles)}/{len(df_raw)} valid SMILES ({len(df_raw)-len(valid_smiles)} dropped)")

In [ ]:
def batch_predict_probs(smiles_list, batch_size=64):
    """Run model inference in batches. Returns (N, 12) numpy array."""
    all_probs = []
    n = len(smiles_list)
    for i in range(0, n, batch_size):
        batch = smiles_list[i:i+batch_size]
        enc = tokenizer(
            batch, return_tensors='pt', truncation=True,
            max_length=512, padding=True,
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        if (i // batch_size) % 5 == 0:
            print(f"  {min(i+batch_size, n)}/{n} compounds scored...", end='\r')
    print(f"  {n}/{n} compounds scored.    ")
    return np.vstack(all_probs)

print("Running batch inference (this takes a few minutes on CPU/MPS)...")
probs_matrix = batch_predict_probs(valid_smiles)  # shape: (N, 12)
print(f"Done. Probabilities shape: {probs_matrix.shape}")

In [ ]:
WEIGHT_ARRAY = np.array([SEVERITY_WEIGHTS[t] for t in TARGET_NAMES])
WEIGHT_SUM   = WEIGHT_ARRAY.sum()

def compute_risk_scores(probs_matrix):
    """Weighted composite score in [0, 1]. Lower = safer."""
    return (probs_matrix * WEIGHT_ARRAY).sum(axis=1) / WEIGHT_SUM

risk_scores = compute_risk_scores(probs_matrix)
print(f"Risk scores — min: {risk_scores.min():.3f}  mean: {risk_scores.mean():.3f}  max: {risk_scores.max():.3f}")

In [ ]:
# === ASSERTION CELL ===
assert probs_matrix.shape == (len(valid_smiles), 12), \
    f"Expected ({len(valid_smiles)}, 12), got {probs_matrix.shape}"
assert not np.isnan(probs_matrix).any(), "NaN probabilities found"
assert (risk_scores >= 0).all() and (risk_scores <= 1).all(), \
    "Risk scores out of [0, 1]"
print("✓ Batch inference and risk scoring assertions passed")